## For positive and negative neuron controls within 80K NGN2 derived neurons

In [61]:
from importlib import reload
import pandas as pd
import sys
sys.path.append('../../../00_helpful_functions')
import helpful_functions as hf
reload(hf)

<module 'helpful_functions' from '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/../../../00_helpful_functions/helpful_functions.py'>

In [139]:
# helpful functions

# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'



variant_control_groups = ["GC_Selvarajan", "GC_Kircher", "GC_Mendelian_variants", "C_positive_heart_CAD", "GC_Atrial_fib", "GC_Mohlke", "GC_Liang"]
variant_groups = ["cardiac_neuro_cava_random"] + variant_control_groups

dnase_control_groups = ['GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled']

synthetic_control_groups = ['C_SLEA']

variant_map_path = '/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_variant_region_map_unique.tsv.gz'
variant_map = pd.read_csv(variant_map_path, sep="\t")
variant_map.columns=['ID', 'Region', 'REF_ID', 'ALT_ID']

variant_map_reference_sequences = variant_map['REF_ID'].to_list()
variant_map_alternative_sequences = variant_map['ALT_ID'].to_list()

def get_category(header):
    """Get the category of the header
    if C_SLEA in tmp_label => synthetic
    elif scramble in header or is DNase group => scrambled
    elif ref_ or alt_ in header => variant
    else element
    """

    label = hf.get_label(header)
    if label in synthetic_control_groups:
        return 'synthetic'

    elif label in dnase_control_groups:
        # all DNase controls used coordinates of hg18
        return 'scrambled'

    elif 'scramble' in header:
        # Check if header is scrambled
        # Info: scrambled is in C_negative_neuron_NP and scramble MK
        # Note: Other cases are not checked with this function
        return 'scrambled'
    else:
        if label in variant_groups and ref_or_alt_in_header(header):
            return 'variant'
        else:
            return 'element'


def is_ref_sequence(header):
    """Check if header is in the REF_ID of the used variant map"""
    # assuming REF_ID holds label information need to split header first
    return header in variant_map_reference_sequences


def is_alt_sequence(header):
    """Check if header is in the ALT_ID of the used variant map"""
    # assuming ALT_ID holds label information need to split header first
    return header in variant_map_alternative_sequences


def ref_or_alt_in_header(header):
    """Checks if "ref_" or "_alt" is in the header"""
    if is_ref_sequence(header) or is_alt_sequence(header):
        return True
    return False


def get_reference_genome(row):
    """If col_category is synthetic or scrambled or in dnase_control_groups, set ref to GRCh37 else GRCh38"""
    row[col_ref] = 'GRCh38'
    if row[col_category] == 'synthetic' or row[col_category] == 'scrambled' or row[col_category] in dnase_control_groups:
        row[col_ref] = 'GRCh37'
    return row


def parse_regions_from_header(row):
    """
    Parse regions from header
    Only possible for the headers you see below
    Not possible for synthetic and scrambled sequences
    22.04: läuft durch getestet nur auf underscore_parsable_headers
    """
    if row[col_category] == 'synthetic' or row[col_category] == 'scrambled':
        return row
    header = row['header']
    label = hf.get_label(header)
    # C_negative_neuron_NP:GW18_PFC_ABC_chr15_89400286_89400556_0.830617698776558
    # C_positive_neuron_NP:GW18_PFC_ABC_chr5_68730744_68731014_6.25979643705865
    # C_positive_neuron_MK:tile_37639_chr6_112501812_112502081_reference_0.828839741594896
    # C_positive_neuron_MK:tile_37639_chr6_112501812_112502081_reference_0.828839741594896
    # C_positive_neuron_MK:tile_37243_chr6_97306567_97306836_A_T_149_0.750659896963247
    NP_group = ['C_negative_neuron_NP', 'C_positive_neuron_NP'] # underscore but without variants
    MK_group = ['C_negative_neuron_MK', 'C_positive_heart_MK', 'C_positive_neuron_MK',  'C_negative_neuron_MK'] # underscore ref: _reference_ alt: _char_char_position
    # double_colon_variants = ['positive_neuron_CD']
    # chrom, ref, alt = "", "", ""
    # start, end, variant_pos = 0, 0, 0
    if label in underscore_parsable_headers:
        # split header at '_chr'
        pre_position = header.split('_chr')
        pre_position = pre_position[1].split('_')
        if len(pre_position) < 3: # unexpected header: throw Value error
            raise ValueError('Unexpected header shape')
        # get chrom start end (group-specific)
        row[col_chr] = f"chr{pre_position[0]}"
        row[col_end] = int(pre_position[2])
        if label in MK_group:
            row[col_start] = int(pre_position[1]) - 1 # turn into 0-based
        elif label in NP_group:
            row[col_start] = int(pre_position[1])
        if len(pre_position) == 7: # variant in header (MK style)
            row[col_variant_class] = 'SNV'
            row[my_col_ref_base] = pre_position[3]
            row[my_col_alt_base] = pre_position[4]
            row[col_variant_pos] = int(pre_position[5])
        else: # for the other rows not following header structure of MK leave them the value they have or set to NA
            try:
                row[my_col_ref_base] = row[my_col_ref_base]
                row[my_col_alt_base] = row[my_col_alt_base]
            except:
                row[my_col_ref_base] = 'NA'
                row[my_col_alt_base] = 'NA'
    return row


def get_start_end_strand_control(row):
    """
    Special cases to set chr, start, end and strand for control sequences from their header (because not in region bed)

    Case: C_negative_neuron_MK: (all C_negative_neuron_MK sequences have "_chr" pattern)
            header: C_negative_neuron_MK:tile_14444_chr15_67066278_67066547_reference__1.1385203581298
                => chr: 15, start: 67066278, end: 67066547, strand: . (no information given)
            variant_header: >C_negative_neuron_MK:tile_36043_chr6_14500968_14501237_G_C_261__0.274168942409752

    Case: C_positive_neuron_MK: (all C_positive_neuron_MK sequences have "_chr" pattern")
            header: C_positive_neuron_MK:tile_35742_chr6_3247831_3248100_reference_0.892141141777512
                => chr: 6, start: 3247831, end: 3248100, strand: . (no information given)

    Case: C_negative_heart_MK: ( all C_negative_heart_MK sequences have "_chr" pattern)
            header: C_negative_heart_MK:tile_6903_chr11_9614045_9614314_reference__0.958461950470297
                => chr: 11, start: 9614045, end: 9614314, strand: . (no information given)

    Case: C_positive_heart_MK: (all C_positive_heart_MK sequences have "_chr" pattern)
            header: C_positive_heart_MK:tile_7939_chr11_65487592_65487861_reference_1.25449216981846
                => chr: 11, start: 65487592, end: 65487861, strand: . (no information given)

    Case: C_negative_neuron_NP: (all C_negative_neuron_NP sequences have "_chr" pattern)
            header: C_negative_neuron_NP:GW18_PFC_ABC_chr15_89400286_89400556_0.830617698776558
                => chr: 15, start: 89400286, end: 89400556, strand: . (no information given)
            additional condition: scrambled_control____2.28928058620308
                => category: scrambled

    Case: C_positive_neuron_NP: (all C_positive_neuron_NP sequences have "_chr" pattern)
            header: C_positive_neuron_NP:GW18_PFC_ABC_chr11_65487667_65487937_5.27702983385667
                => chr: 11, start: 65487667, end: 65487937, strand: . (no information given)

    Case: C_positive_neuron_CD: headers have "::chr" pattern and delimited by "-mean_ratio"
            header: C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A::chr4:155359187-155359457-mean_ratio2.42
                => chr: 4, start: 155359187, end: 155359457, strand: . (no information given)
            additional condition: C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_94-mean_ratio2.31

    """

    if row[col_category] == 'synthetic' or row[col_category] == 'scrambled':
        return row
    name = row[col_name]
    # print(name) # TODO: remove debug

    if name.startswith('C_negative_heart_MK') or name.startswith('C_negative_neuron_MK') or name.startswith('C_positive_heart_MK') or name.startswith('C_positive_neuron_MK'):
        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row

        row[col_class] = 'element inactive control'
        if 'positive_neuron' in name:
            row[col_class] = 'element active control'
        region_info = '_'.join(name.split('_chr')[1].split('_')[:3]) # 11_9614045_9614314
        row[col_category] = 'element'
        row[col_chr] = f"chr{region_info.split('_')[0]}"
        row[col_start] = int(region_info.split('_')[1]) -1 # turn into 0-based
        row[col_end] = int(region_info.split('_')[2])
        row[col_strand] = '.'
        row[col_source] = 'Michael Kosicki'
        if len(name.split('_chr')[1].split("_")) == 7:
            row[col_variant_class] = 'SNV'
            row[col_category] = 'variant'
            row[col_class] = 'variant negative control'
            row[my_col_ref_base] = name.split('_chr')[1].split("_")[3]
            row[my_col_alt_base] = name.split('_chr')[1].split("_")[4]
            row[col_variant_pos] = int(name.split('_chr')[1].split("_")[5]) - 1
            row[col_info] = row[col_info] + '; No reference given'
            row[col_allele] = 'alt'
            if 'positive_neuron' in name:
                row[col_class] = 'variant positive control'

    elif name.startswith('C_negative_neuron_NP') or name.startswith('C_positive_neuron_NP'):
        row[col_class] = 'element inactive control'
        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row
        if 'positive_neuron' in name:
            row[col_class] = 'element active control'
        region_info = '_'.join(name.split('_chr')[1].split('_')[:3]) # 11_9614045_9614314
        # print(region_info)
        row[col_chr] = f"chr{region_info.split('_')[0]}"
        row[col_start] = int(region_info.split('_')[1])
        row[col_end] = int(region_info.split('_')[2])
        row[col_strand] = '.'
        row[col_source] = 'Nick Page'

    elif name.startswith('C_positive_neuron_CD'):
        row[col_class] = 'element active control'
        if 'NA_NA_NA' in name:
            row[col_category] = 'scrambled'
            row[col_chr] = 'NA'
            row[col_start] = 'NA'
            row[col_end] = 'NA'
            row[col_strand] = 'NA'
            row[col_info] = row[col_info] + '; Information not restorable from header'
            return row
        region_info = name.split('::chr')[1].split('-mean_ratio')[0] # 4:155359187-155359457
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'
        row[col_source] = 'Chengyu Deng'
        variant_info = name.split('C_positive_neuron_CD:')[1].split('::chr')[0].split('_')
        if len(variant_info) >= 6:
            row[col_variant_class] = 'SNV'
            row[col_category] = 'variant'
            row[col_class] = 'variant positive control'
            row[col_allele] = 'alt'
            reference_info = 'No reference assigned'
            row[my_col_ref_base] = variant_info[2]
            row[my_col_alt_base] = variant_info[3]
            row[col_variant_pos] = int(variant_info[5])
            # todo: variant_pos
            # todo: ref_base and alt_base
            if '_ref_' in name:
                row[col_allele] = 'ref'
                reference_info = 'No alternative assigned'
            row[col_info] = row[col_info] + f'; {reference_info}'
            row[col_info] = row[col_info] + '; Already tested in another MPRA'

    elif name.startswith('GC_DNase_positive:') or name.startswith('GC_DNase_negative_brain:') or name.startswith('GC_DNase_negative_blood:'):
        row[col_ref] = 'GRCh37'
        row[col_class] = 'element inactive control'
        region_info = name.split(':chr')[1].split('_active_count_')[0]
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'
        # add info that coordinates of GRCh37 are used
        row[col_info] = row[col_info] + '; Coordinates are based on GRCh37'
    return row


# dict of chr number to refseq chromosome number
chrom_2_refseq = {
    "chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}

# get args with click: input file, seperator, ids of chr, pos, ref, alt in string

def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) # I think everything is now 0-based
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'


def get_spdi(row):
    """
    Returns the SPDI identifier for the given variant using start and variant_pos
    Assumption: allele need to be set beforehands
    """
    # TODO: add case for controls
    if not (hf.is_alternative(row[col_allele])) and (row[my_col_ref_base] and row[my_col_alt_base]):
        row['SPDI'] = 'NA'
        return row
    # identify the variant chrom-pos-ref-alt pattern
    chrom_pos_ref_alt = f'{row[col_chr]}-{row[col_start]+row[col_variant_pos]}-{row[my_col_ref_base]}-{row[my_col_alt_base]}'
    if chrom_pos_ref_alt == "NA":
        raise ValueError('Variant pattern could not be found')
    # create the SPDI identifier
    row[col_SPDI] = create_speedy_chromosomes(chrom_pos_ref_alt, seperator='-', indices=[0,1,2,3])
    return row


In [140]:
pre_metadata_df = hf.fasta_to_dataframe('/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_design_no_duplicates_sequence_and_header_with_adapter_no_brackets_no_collisions.fa', columns=[col_name, col_sequence])
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))
pre_metadata_df
underscore_parsable_headers = ['C_negative_neuron_NP', 'C_positive_neuron_NP', 'C_positive_neuron_MK',  'C_negative_neuron_MK', 'C_positive_heart_MK', 'C_negative_heart_MK', 'C_positive_neuron_CD']

# # filter for underscore parsable headers
pre_metadata_df_filtered = pre_metadata_df.loc[pre_metadata_df['tmp_label'].isin(underscore_parsable_headers)]
pre_metadata_df_filtered # 971

# add the columns of the metadata file
pre_metadata_df_filtered[col_category] = 'NA'
pre_metadata_df_filtered[col_class] = 'NA'
pre_metadata_df_filtered[col_source] = 'NA'
pre_metadata_df_filtered[col_ref] = 'NA'
pre_metadata_df_filtered[col_chr] = 'NA'
pre_metadata_df_filtered[col_start] = 'NA'
pre_metadata_df_filtered[col_end] = 'NA'
pre_metadata_df_filtered[col_strand] = 'NA'
pre_metadata_df_filtered[col_variant_class] = 'NA'
pre_metadata_df_filtered[col_variant_pos] = 'NA'
pre_metadata_df_filtered[col_SPDI] = 'NA'
pre_metadata_df_filtered[col_allele] = 'NA'
pre_metadata_df_filtered[col_info] = 'NA'


pre_metadata_df_filtered[col_category] = pre_metadata_df_filtered[col_name].apply(get_category)
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(get_reference_genome, axis=1)

# parse regions from header
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(get_start_end_strand_control, axis=1)
pre_metadata_df_filtered

# add SPDI
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(lambda row: get_spdi(row), axis=1)

# remove adapter from sequence (15bp of start and end):
pre_metadata_df_filtered[col_sequence] = pre_metadata_df_filtered[col_sequence].apply(lambda x: x[15:-15])

/tmp/ipykernel_313048/2314396430.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pre_metadata_df_filtered[col_category] = 'NA'
/tmp/ipykernel_313048/2314396430.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pre_metadata_df_filtered[col_class] = 'NA'
/tmp/ipykernel_313048/2314396430.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.or

In [151]:
interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

# Write DataFrame to TSV file
pre_metadata_df_filtered[interesting_columns].to_csv('/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/neuro_controls.metadata.tmp.tsv.gz', sep='\t', index=False, na_rep='NA', compression='gzip')
import os
os.system('zcat /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/neuro_controls.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/neuro_controls.metadata.tsv.gz')

0